# 유튜브 댓글 데이터 전처리 파이프라인
감정 키워드 기반 영상 추천 AI 모델을 위한 데이터 전처리

In [1]:
# 셀 1: 라이브러리 임포트
import pandas as pd
import numpy as np
import re
import string
import warnings
import pickle
from konlpy.tag import Okt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
print('라이브러리 임포트 완료')

라이브러리 임포트 완료


In [2]:
# 셀 2: 데이터 로드 및 병합 (w3, w4 lab 패턴)
files = {
    '뉴진스': '뉴진스_좋아요포함_댓글50개_영상50개_20260524_145120.xlsx',
    '월드컵': '월드컵_좋아요포함_댓글50개_영상50개_20260524_211337.xlsx',
    '이스라엘': '이스라엘_좋아요포함_댓글50개_영상50개_20260524_181319.xlsx',
    '인공지능': '인공지능_좋아요포함_댓글50개_영상50개_20260522_101633.xlsx',
}

dfs = []
for category, filename in files.items():
    temp = pd.read_excel(filename)
    temp['category'] = category
    dfs.append(temp)
    print(f'{category}: {temp.shape[0]}건 로드')

# pd.concat()으로 병합 (w4 lab 패턴)
df = pd.concat(dfs, ignore_index=True)
print(f'\n전체 병합 완료: {df.shape[0]}건, 컬럼: {list(df.columns)}')

뉴진스: 2500건 로드


월드컵: 2430건 로드


이스라엘: 2410건 로드


인공지능: 2370건 로드

전체 병합 완료: 9710건, 컬럼: ['Video Title', 'Video URL', 'Likes', 'Comment', 'category']


In [3]:
# 셀 3: 데이터 탐색 EDA (w3, w4 lab 패턴)
print('=== df.shape ===')
print(df.shape)

print('\n=== df.info() ===')
df.info()

print('\n=== df.head() ===')
display(df.head())

print('\n=== df.describe() ===')
display(df.describe())

print('\n=== 카테고리별 건수 ===')
print(df['category'].value_counts())

# 카테고리별 영상 수 (w3 lab - groupby 패턴)
print('\n=== 카테고리별 영상 수 ===')
print(df.groupby('category')['Video Title'].nunique())

print('\n=== 결측치 확인 ===')
print(df.isnull().sum())

=== df.shape ===
(9710, 5)

=== df.info() ===
<class 'pandas.DataFrame'>
RangeIndex: 9710 entries, 0 to 9709
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Video Title  9710 non-null   str  
 1   Video URL    9710 non-null   str  
 2   Likes        9710 non-null   str  
 3   Comment      9710 non-null   str  
 4   category     9710 non-null   str  
dtypes: str(5)
memory usage: 379.4 KB

=== df.head() ===


,Video Title,Video URL,Likes,Comment,category
0,NewJeans (뉴진스) 'OMG' Official MV (Performance ...,https://www.youtube.com/watch?v=sVTy_wmn5SU&li...,"다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시",민희진님은 절대 ㅈㅅ 안한다고 했습니다.\n️️️️️️️️️️\n이 댓글이 삭제된다...,뉴진스
1,NewJeans (뉴진스) 'OMG' Official MV (Performance ...,https://www.youtube.com/watch?v=sVTy_wmn5SU&li...,"다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시",뉴진스 데뷔 미룬 것도 르세라핌이 5.2에 데뷔해야해서 그런거냐ㅜ방시혁 정신 좀 차...,뉴진스
2,NewJeans (뉴진스) 'OMG' Official MV (Performance ...,https://www.youtube.com/watch?v=sVTy_wmn5SU&li...,"다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시",NewJeans really manifested their concept. They...,뉴진스
3,NewJeans (뉴진스) 'OMG' Official MV (Performance ...,https://www.youtube.com/watch?v=sVTy_wmn5SU&li...,"다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시","NewJeans will always be five. Danielle, you wi...",뉴진스
4,NewJeans (뉴진스) 'OMG' Official MV (Performance ...,https://www.youtube.com/watch?v=sVTy_wmn5SU&li...,"다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시",희진좌 도대체 어떤 싸움을 해오신겁니까... 그냥 사이비도 아니고 정계까지 엮여있는...,뉴진스



=== df.describe() ===


,Video Title,Video URL,Likes,Comment,category
count,9710,9710,9710,9710,9710
unique,196,197,196,9647,4
top,Super Shy,https://www.youtube.com/watch?v=sVTy_wmn5SU&li...,"다른 사용자 5,054명과 함께 이 동영상에 좋아요 표시",NEWJEANS IS FIVE,뉴진스
freq,100,50,100,7,2500



=== 카테고리별 건수 ===
category
뉴진스     2500
월드컵     2430
이스라엘    2410
인공지능    2370
Name: count, dtype: int64

=== 카테고리별 영상 수 ===
category
뉴진스     49
월드컵     49
이스라엘    50
인공지능    48
Name: Video Title, dtype: int64

=== 결측치 확인 ===
Video Title    0
Video URL      0
Likes          0
Comment        0
category       0
dtype: int64


In [4]:
# 셀 4: Likes 컬럼 숫자 추출 (w4 lab - 컬럼 변환 패턴)
print('변환 전 Likes 샘플:')
print(df['Likes'].head(3).tolist())

# 정규표현식으로 숫자 추출: "다른 사용자 2,490,682명과 함께..." → 2490682
# "좋아요"만 있는 경우(숫자 없음) → 0으로 처리
extracted = df['Likes'].str.extract(r'([\d,]+)명')[0]
df['Likes'] = (
    extracted
    .str.replace(',', '', regex=False)
    .astype(float)
    .fillna(0)
    .astype(int)
)

print(f'\n변환 후 Likes 샘플:')
print(df['Likes'].head(3).tolist())
print(f'숫자 추출 실패(0으로 대체): {(df["Likes"] == 0).sum()}건')
print(f'\nLikes 통계:\n{df["Likes"].describe()}')

변환 전 Likes 샘플:
['다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시', '다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시', '다른 사용자 2,490,682명과 함께 이 동영상에 좋아요 표시']

변환 후 Likes 샘플:
[2490682, 2490682, 2490682]
숫자 추출 실패(0으로 대체): 30건

Likes 통계:
count    9.710000e+03
mean     2.346665e+05
std      4.836621e+05
min      0.000000e+00
25%      8.966000e+03
50%      1.536300e+04
75%      1.217710e+05
max      3.085098e+06
Name: Likes, dtype: float64


In [5]:
# 셀 5: 중복 제거 (w4 lab - drop 패턴)
before_count = len(df)
dup_mask = df.duplicated(subset=['Comment', 'Video URL'], keep='first')
print(f'중복 댓글 수: {dup_mask.sum()}건')

df = df.drop_duplicates(subset=['Comment', 'Video URL'], keep='first').reset_index(drop=True)
after_count = len(df)
print(f'중복 제거: {before_count}건 → {after_count}건 (제거: {before_count - after_count}건)')

중복 댓글 수: 0건
중복 제거: 9710건 → 9710건 (제거: 0건)


In [6]:
# 셀 6: 텍스트 클리닝 (w5 lab - Cleaning 패턴)
def clean_text(text):
    """댓글 텍스트 클리닝 함수"""
    if not isinstance(text, str):
        return ''
    # HTML 태그 제거 (w5 lab의 BeautifulSoup 패턴 대신 regex 사용)
    text = re.sub(r'<.*?>', '', text)
    # URL 제거
    text = re.sub(r'https?://\S+', '', text)
    # 이모지/특수문자 제거: 한글, 영어, 숫자, 기본 공백만 유지
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    # 줄바꿈 및 과도한 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    # 영어 소문자 통일 (w5 lab 패턴)
    text = text.lower()
    return text

# 클리닝 전후 비교 샘플
print('=== 클리닝 전후 비교 ===')
sample_indices = [0, 1, 2]
for i in sample_indices:
    print(f'\n[{i}] Before: {df["Comment"].iloc[i][:80]}...')
    print(f'[{i}] After : {clean_text(df["Comment"].iloc[i])[:80]}...')

df['Comment_clean'] = df['Comment'].apply(clean_text)
print(f'\n텍스트 클리닝 완료: {len(df)}건')

=== 클리닝 전후 비교 ===

[0] Before: 민희진님은 절대 ㅈㅅ 안한다고 했습니다.
️️️️️️️️️️
이 댓글이 삭제된다면 하이브에서 삭제하는 것입니다. 좋아요 눌러서 위로 올려주세요...
[0] After : 민희진님은 절대 안한다고 했습니다 이 댓글이 삭제된다면 하이브에서 삭제하는 것입니다 좋아요 눌러서 위로 올려주세요...

[1] Before: 뉴진스 데뷔 미룬 것도 르세라핌이 5.2에 데뷔해야해서 그런거냐ㅜ방시혁 정신 좀 차려 제발...
[1] After : 뉴진스 데뷔 미룬 것도 르세라핌이 5 2에 데뷔해야해서 그런거냐 방시혁 정신 좀 차려 제발...

[2] Before: NewJeans really manifested their concept. They are now like a beautiful memory o...
[2] After : newjeans really manifested their concept they are now like a beautiful memory of...

텍스트 클리닝 완료: 9710건


In [7]:
# 셀 7: 언어 감지 및 토큰화 (w5 lab - Tokenization & Lemmatization 패턴)
okt = Okt()
lemmatizer = WordNetLemmatizer()

def detect_language(text):
    """한/영 판별: 한글 문자 비율 기반"""
    if not text:
        return 'ko'
    korean_chars = re.findall(r'[가-힣]', text)
    total_chars = re.findall(r'[가-힣a-zA-Z]', text)
    if len(total_chars) == 0:
        return 'ko'
    return 'ko' if len(korean_chars) / len(total_chars) > 0.5 else 'en'

def tokenize_text(text):
    """언어에 따라 토큰화 수행"""
    if not text:
        return []
    lang = detect_language(text)
    if lang == 'ko':
        # 한국어: Okt 형태소 분석 + 어간 추출
        tokens = okt.morphs(text, stem=True)
    else:
        # 영어: word_tokenize + lemmatization (w5 lab 패턴)
        tokens = word_tokenize(text)
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# 언어 분포 확인
df['lang'] = df['Comment_clean'].apply(detect_language)
print('=== 언어 분포 ===')
print(df.groupby('category')['lang'].value_counts())

# 토큰화 실행
print('\n토큰화 진행 중...')
df['tokens'] = df['Comment_clean'].apply(tokenize_text)
print(f'토큰화 완료: {len(df)}건')
print(f'\n토큰 샘플:')
for i in range(3):
    print(f'  [{i}] {df["tokens"].iloc[i][:10]}')

=== 언어 분포 ===
category  lang
뉴진스       ko      1315
          en      1185
월드컵       ko      2394
          en        36
이스라엘      ko      2397
          en        13
인공지능      ko      2359
          en        11
Name: count, dtype: int64

토큰화 진행 중...


토큰화 완료: 9710건

토큰 샘플:
  [0] ['민희진', '님', '은', '절대', '안', '하다', '하다', '이', '댓글', '이']
  [1] ['뉴진스', '데뷔', '미루다', '것', '도', '르', '세라핌', '이', '5', '2']
  [2] ['newjeans', 'really', 'manifested', 'their', 'concept', 'they', 'are', 'now', 'like', 'a']


In [8]:
# 셀 8: 불용어 제거 (w5 lab - Removing Stopwords 패턴)

# 한국어 불용어 리스트 (조사, 접속사, 의미 없는 단어)
korean_stopwords = [
    '의', '가', '이', '은', '는', '을', '를', '에', '에서', '도', '로', '으로',
    '와', '과', '하다', '있다', '되다', '이다', '않다', '없다', '같다',
    '그', '저', '것', '수', '등', '들', '및', '더', '좀', '잘', '못',
    '한', '두', '세', '네', '다', '또', '또한', '그리고', '하지만', '그러나',
    '때문', '위해', '대해', '통해', '따라', '관련', '대한',
    '너무', '정말', '진짜', '매우', '아주', '참', '꽤',
    '거', '게', '데', '뭐', '왜', '어떻게', '얼마나',
    'ㅋ', 'ㅋㅋ', 'ㅋㅋㅋ', 'ㅎ', 'ㅎㅎ', 'ㅎㅎㅎ', 'ㅠ', 'ㅠㅠ', 'ㅜ', 'ㅜㅜ',
    'ㄹㅇ', 'ㅇㅇ', 'ㄱㄱ', 'ㅇㅈ', 'ㅂㅂ',
]

# 영어 불용어 (w5 lab 동일 패턴)
english_stopwords = set(stopwords.words('english'))

def remove_stopwords(tokens, lang):
    """언어에 맞는 불용어 제거"""
    if lang == 'ko':
        return [t for t in tokens if t not in korean_stopwords and len(t) > 1]
    else:
        return [t for t in tokens if t not in english_stopwords and len(t) > 1]

df['tokens_clean'] = df.apply(lambda row: remove_stopwords(row['tokens'], row['lang']), axis=1)

print('=== 불용어 제거 전후 토큰 수 비교 ===')
print(f'제거 전 평균 토큰 수: {df["tokens"].apply(len).mean():.1f}')
print(f'제거 후 평균 토큰 수: {df["tokens_clean"].apply(len).mean():.1f}')
print(f'\n샘플:')
for i in range(3):
    print(f'  [{i}] Before: {df["tokens"].iloc[i][:8]}')
    print(f'       After : {df["tokens_clean"].iloc[i][:8]}')

=== 불용어 제거 전후 토큰 수 비교 ===
제거 전 평균 토큰 수: 18.7
제거 후 평균 토큰 수: 10.9

샘플:
  [0] Before: ['민희진', '님', '은', '절대', '안', '하다', '하다', '이']
       After : ['민희진', '절대', '댓글', '삭제', '하이브', '삭제', '좋다', '누르다']
  [1] Before: ['뉴진스', '데뷔', '미루다', '것', '도', '르', '세라핌', '이']
       After : ['뉴진스', '데뷔', '미루다', '세라핌', '데뷔', '그런', '방시혁', '정신']
  [2] Before: ['newjeans', 'really', 'manifested', 'their', 'concept', 'they', 'are', 'now']
       After : ['newjeans', 'really', 'manifested', 'concept', 'like', 'beautiful', 'memory', 'past']


In [9]:
# 셀 9: 짧은/의미없는 댓글 필터링
before_count = len(df)
df = df[df['tokens_clean'].apply(len) >= 2].reset_index(drop=True)
after_count = len(df)
print(f'짧은 댓글 필터링: {before_count}건 → {after_count}건 (제거: {before_count - after_count}건)')
print(f'\n카테고리별 잔여 건수:')
print(df['category'].value_counts())

짧은 댓글 필터링: 9710건 → 9466건 (제거: 244건)

카테고리별 잔여 건수:
category
뉴진스     2402
월드컵     2391
이스라엘    2340
인공지능    2333
Name: count, dtype: int64


In [10]:
# 셀 10: 영상 단위 댓글 집계 (w3 lab - groupby 패턴)
# 각 영상의 모든 클린 토큰을 합쳐 하나의 문서(공백 구분 문자열)로 구성
video_df = (
    df.groupby(['Video URL', 'Video Title', 'category', 'Likes'])
    .agg({
        'tokens_clean': lambda x: ' '.join([' '.join(tokens) for tokens in x]),
        'Comment': 'count'
    })
    .rename(columns={'tokens_clean': 'all_tokens', 'Comment': 'comment_count'})
    .reset_index()
)

print(f'영상 단위 집계 완료: {len(video_df)}개 영상')
print(f'\n카테고리별 영상 수:')
print(video_df['category'].value_counts())
print(f'\n영상별 댓글 수 통계:')
print(video_df['comment_count'].describe())
display(video_df.head())

영상 단위 집계 완료: 197개 영상

카테고리별 영상 수:
category
뉴진스     50
이스라엘    50
월드컵     49
인공지능    48
Name: count, dtype: int64

영상별 댓글 수 통계:
count    197.000000
mean      48.050761
std        4.094230
min       20.000000
25%       48.000000
50%       49.000000
75%       50.000000
max       50.000000
Name: comment_count, dtype: float64


,Video URL,Video Title,category,Likes,all_tokens,comment_count
0,https://www.youtube.com/watch?v=-72Dmj3oRd8&pp...,석학들이 머리 맞대고 만든 음식 월드컵,월드컵,9242,오프닝 편집 까지 잡다 전문 시청 결국 오프닝 까지 49 54 쌍베 뻔하다 떡볶이 ...,50
1,https://www.youtube.com/watch?v=0DIvs70EsBk&pp...,17년만에 야구 월드컵 본선 진출 성공해서 초흥분 상태가 된 야구팬들 ㅋㅋㅋㅋㅋㅋ ...,월드컵,11058,보경 원래 은우 탈세 박보검 교체 일본인 웃기다 아니다 일단 가져가다 사람 생각 똑...,49
2,https://www.youtube.com/watch?v=0DwFOQeUkuI&pp...,"가나, 우루과이 잡으시고, 바지적삼, 다 적시셨네 / 스브스뉴스",월드컵,88650,영상 보다 따다 가나 고맙다 오늘 부터 고향 아티 지기 골킥 거의 30초 차다 골킥...,50
3,https://www.youtube.com/watch?v=0KDosjF0iYM&pp...,"AI 특이점, 5년 안에 온다고? 프콘도 깜짝 놀란 과학자들의 진짜 AI 썰 (fe...",인공지능,25736,댓글 이벤트 당첨 발표 댓글 이벤트 참여 해주다 모든 에게 진심 감사 드리다 당첨 ...,49
4,https://www.youtube.com/watch?v=0c7zGU2C2mM&li...,Super Shy,뉴진스,730749,give back newjeans song feel like yesterday ca...,50


In [11]:
# 셀 11: TF-IDF 인코딩 (w5 lab - Encoding 패턴)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(video_df['all_tokens'])

feature_names = tfidf_vectorizer.get_feature_names_out()
print(f'TF-IDF 행렬 차원: {tfidf_matrix.shape} (영상 수 × feature 수)')
print(f'상위 20개 feature: {list(feature_names[:20])}')

# TF-IDF 상위 키워드를 카테고리별로 확인
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
tfidf_df['category'] = video_df['category'].values

print('\n=== 카테고리별 TF-IDF 상위 키워드 ===')
for cat in ['뉴진스', '월드컵', '이스라엘', '인공지능']:
    cat_mean = tfidf_df[tfidf_df['category'] == cat].drop(columns='category').mean()
    top_words = cat_mean.nlargest(10)
    print(f'\n[{cat}] {list(top_words.index)}')

TF-IDF 행렬 차원: (197, 5000) (영상 수 × feature 수)
상위 20개 feature: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '100', '1000', '100년', '100만', '100분', '100회', '10년', '10분', '11']

=== 카테고리별 TF-IDF 상위 키워드 ===



[뉴진스] ['뉴진스', 'newjeans', '노래', 'miss', 'new', 'song', 'never', 'jean', 'danielle', 'like']

[월드컵] ['월드컵', '보다', '선수', '경기', '축구', '손흥민', '이강인', '감독', '좋다', '웃기다']

[이스라엘] ['전쟁', '이스라엘', '이란', '미국', '네타냐후', '트럼프', '보다', '나라', '국민', '악마']



[인공지능] ['ai', '인간', '교수', '보다', '시대', '사람', '가다', '판사', '생각', '감사하다']


In [12]:
# 셀 12: 전처리 결과 저장

# 1) 댓글 단위 전처리 결과
comments_save = df[['Video Title', 'Video URL', 'Likes', 'Comment', 'category',
                     'Comment_clean', 'lang', 'tokens_clean']].copy()
# 토큰 리스트를 공백 구분 문자열로 변환하여 저장
comments_save['tokens_clean'] = comments_save['tokens_clean'].apply(lambda x: ' '.join(x))
comments_save.to_csv('preprocessed_comments.csv', index=False, encoding='utf-8-sig')
print(f'preprocessed_comments.csv 저장 완료: {len(comments_save)}행')

# 2) 영상 단위 집계 결과
video_df.to_csv('video_documents.csv', index=False, encoding='utf-8-sig')
print(f'video_documents.csv 저장 완료: {len(video_df)}행')

# 3) TF-IDF 행렬
with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)
print(f'tfidf_matrix.pkl 저장 완료: {tfidf_matrix.shape}')

# 4) TF-IDF Vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print('tfidf_vectorizer.pkl 저장 완료')

# 전처리 전후 비교 요약
print('\n' + '='*50)
print('전처리 파이프라인 완료 요약')
print('='*50)
print(f'원본 데이터: {sum(len(d) for d in dfs)}건')
print(f'전처리 후 댓글: {len(df)}건')
print(f'영상 단위 문서: {len(video_df)}개')
print(f'TF-IDF 차원: {tfidf_matrix.shape}')
print(f'\n출력 파일:')
print(f'  - preprocessed_comments.csv')
print(f'  - video_documents.csv')
print(f'  - tfidf_matrix.pkl')
print(f'  - tfidf_vectorizer.pkl')

preprocessed_comments.csv 저장 완료: 9466행
video_documents.csv 저장 완료: 197행
tfidf_matrix.pkl 저장 완료: (197, 5000)
tfidf_vectorizer.pkl 저장 완료

전처리 파이프라인 완료 요약
원본 데이터: 9710건
전처리 후 댓글: 9466건
영상 단위 문서: 197개
TF-IDF 차원: (197, 5000)

출력 파일:
  - preprocessed_comments.csv
  - video_documents.csv
  - tfidf_matrix.pkl
  - tfidf_vectorizer.pkl
